In [21]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [22]:
df = pd.read_csv("../data/processed/telco_cleaned.csv")

print(df.shape)
df.head()

(7032, 20)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [23]:
df["tenure_group"] = pd.cut(
    df["tenure"],
    bins=[0,12,24,48,72],
    labels=["0-12","12-24","24-48","48+"]
)

In [24]:
services = [
    "PhoneService",
    "MultipleLines",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

df["service_count"] = (df[services] == "Yes").sum(axis=1)

df["service_count"].describe()

count    7032.000000
mean        3.363339
std         2.062067
min         0.000000
25%         1.000000
50%         3.000000
75%         5.000000
max         8.000000
Name: service_count, dtype: float64

In [25]:
df["high_monthly_charges"] = (df["MonthlyCharges"] > 80).astype(int)

df["high_monthly_charges"].value_counts()

high_monthly_charges
0    4367
1    2665
Name: count, dtype: int64

In [26]:
df["auto_payment"] = df["PaymentMethod"].isin([
    "Bank transfer (automatic)",
    "Credit card (automatic)"
]).astype(int)

In [27]:
df["fiber_customer"] = (df["InternetService"] == "Fiber optic").astype(int)

In [28]:
df["long_contract"] = df["Contract"].isin([
    "One year",
    "Two year"
]).astype(int)

In [29]:
df["customer_lifetime_value"] = df["tenure"] * df["MonthlyCharges"]

df["customer_lifetime_value"].describe()

count    7032.000000
mean     2283.147248
std      2264.703327
min        18.800000
25%       397.800000
50%      1394.575000
75%      3791.250000
max      8550.000000
Name: customer_lifetime_value, dtype: float64

In [30]:
df["avg_monthly_spend"] = df["TotalCharges"] / (df["tenure"] + 1)

df["avg_monthly_spend"].describe()

count    7032.000000
mean       59.083067
std        30.514438
min         9.183333
25%        26.225944
50%        61.070387
75%        84.877538
max       118.969863
Name: avg_monthly_spend, dtype: float64

In [31]:
support_services = [
    "OnlineSecurity",
    "OnlineBackup",
    "TechSupport",
    "DeviceProtection"
]

df["support_dependency"] = (df[support_services] == "Yes").sum(axis=1)

df["support_dependency"].describe()

count    7032.000000
mean        1.265358
std         1.286277
min         0.000000
25%         0.000000
50%         1.000000
75%         2.000000
max         4.000000
Name: support_dependency, dtype: float64

In [32]:
df["Churn"] = df["Churn"].map({
    "No":0,
    "Yes":1
})

In [33]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

In [34]:
cat_cols = X.select_dtypes(include="object").columns
cat_cols

Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='object')

In [35]:
X = pd.get_dummies(
    X,
    columns=cat_cols,
    drop_first=True
)

In [36]:
num_cols = X.select_dtypes(include=["int64","float64"]).columns

scaler = StandardScaler()

X[num_cols] = scaler.fit_transform(X[num_cols])

In [37]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [38]:
X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)

y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

In [39]:
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_train shape: (5625, 39)
y_train shape: (5625,)
X_test shape: (1407, 39)
y_test shape: (1407,)
